In [1]:
import pandas as pd
import numpy as np

# === Load raw data ===
df = pd.read_csv("data/customer_data.csv")
print(f"Loaded {len(df)} rows.")

# === Step 1: Fix column types ===
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')

# === Step 2: Standardize categorical labels ===
df['city'] = df['city'].str.strip().str.title().replace({'Bengaluru': 'Bangalore'})
df['gender'] = df['gender'].str.strip().str.title().replace({'M': 'Male', 'F': 'Female'})
df['segment'] = df['segment'].str.strip().str.title()

# === Step 3: Missing-value strategy (documented) ===
# age: ~4% missing, roughly random -> median fill, flagged
df['age_was_missing'] = df['age'].isna()
df['age'] = df['age'].fillna(df['age'].median())

# income: ~12% missing + placeholder outlier (~10,000,000) -> treat outlier as missing, then median fill, flagged
df['income'] = df['income'].where(df['income'] < 1_000_000, np.nan)
df['income_was_missing'] = df['income'].isna()
df['income'] = df['income'].fillna(df['income'].median())

# satisfaction_score: ordinal 1-5, ~3% missing -> mode fill
df['satisfaction_score'] = df['satisfaction_score'].fillna(df['satisfaction_score'].mode()[0])

# signup_date: left as NaT where missing -> too risky to guess a date, decision documented not fixed

# === Step 4: Auxiliary table merge ===
# No auxiliary tables in this practice dataset - single source only.

# === Step 5: Derived columns ===
df['age_band'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 60, 100],
                         labels=['18-25', '26-35', '36-45', '46-60', '60+'])
df['signup_year'] = df['signup_date'].dt.year
df['signup_month'] = df['signup_date'].dt.month
df['income_band'] = pd.cut(df['income'], bins=[0, 30000, 60000, 100000, np.inf],
                            labels=['Low', 'Mid', 'High', 'Very High'])

# === Step 6: Drop exact duplicates and save ===
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Dropped {before - after} exact duplicate rows. Final row count: {after}")



df.to_csv("data/customer_data_cleaned.csv", index=False)
print("Cleaned dataset saved to data/customer_data_cleaned.csv")

Loaded 505 rows.
Dropped 5 exact duplicate rows. Final row count: 500
Cleaned dataset saved to data/customer_data_cleaned.csv


## Missing Value Strategy (Task 4)

| Column | Missing % | Strategy | Reason |
|---|---|---|---|
| age | ~4% | Median fill, flagged in `age_was_missing` | Roughly random missingness; median resists outlier skew |
| income | ~12% (+ outlier) | Outlier treated as missing, then median fill, flagged in `income_was_missing` | Mean would be distorted by the placeholder; median is robust |
| satisfaction_score | ~3% | Mode fill | Ordinal rating column; most frequent value is the sensible fill |
| signup_date | ~2% | Left as missing (NaT), not filled | Too risky to guess a date; documented rather than invented |

**Known open issue (not fixed):** `customer_id` reuse on different-looking records (e.g. IDs 1073, 1104, 1124, 1155) was identified in Task 2 but not resolved here — it needs investigation rather than a quick fix, since dropping a row risks losing real data.